# Chapter 5 — Loading Pretrained GPT-2 Weights

In the previous chapters we built a complete GPT architecture from scratch.  
The model works, but its weights are random — it produces gibberish.  
In this notebook we fix that by transplanting the weights that OpenAI trained  
on ~40 GB of web text into our custom `GPTModel` class.

### Why not just use the HuggingFace model directly?
Because the point is to understand every byte of the architecture.  
Loading pretrained weights into *our* code is a strict test that our  
implementation matches GPT-2 exactly — if any layer is wrong, generation breaks.

### The two main challenges
1. **Conv1D vs nn.Linear** — HuggingFace's GPT-2 uses a custom `Conv1D` layer  
   whose weight matrix is stored *transposed* relative to `nn.Linear`.  
   We must call `.T` on every attention and MLP weight before copying it.
2. **Combined QKV** — HuggingFace stores Q, K, V as one fused `c_attn` matrix  
   of shape `[768, 2304]`.  We must split it into our three separate projections.

## 1. Imports

We add the project root to `sys.path` so that `model.py` (one directory up)  
can be imported directly without installing it as a package.

In [ ]:
import sys                                         # standard library: manipulate the Python module search path
import os                                          # standard library: OS-agnostic path operations
sys.path.insert(0, os.path.abspath(".."))          # prepend project root so 'model.py' is importable from here

import torch                                       # PyTorch: the core deep learning framework
from transformers import GPT2Model                 # HuggingFace GPT2Model (transformer only, no LM head wrapper)
import tiktoken                                    # OpenAI BPE tokenizer — same vocab GPT-2 was trained with

from model import GPTModel, generate               # our scratch-built model class and greedy decode function

## 2. GPT-2 Small Configuration

GPT-2 'small' has **124 million parameters**.  
All hyperparameters below are fixed by the original OpenAI release — we must  
match them exactly or the weight shapes will not align.

Key difference from our training experiments: `qkv_bias=True`.  
GPT-2 adds a bias term to every Q, K, V projection; many newer models  
(e.g. LLaMA) remove it to save parameters and memory.

In [ ]:
GPT_CONFIG_124M = {               # configuration dict consumed by our GPTModel constructor
    "vocab_size":     50257,      # BPE vocabulary size: 50,256 merge rules + special <|endoftext|> token
    "context_length": 1024,       # maximum number of tokens the model attends to at once
    "emb_dim":        768,        # hidden / embedding dimension shared by all layers
    "n_heads":        12,         # number of attention heads; each head has dim = 768 / 12 = 64
    "n_layers":       12,         # depth: 12 stacked transformer blocks
    "drop_rate":      0.1,        # dropout probability (only active during training; we use eval mode here)
    "qkv_bias":       True,       # GPT-2 uses bias in Q/K/V projections — must be True to load its weights
}

## 3. Downloading Pretrained Weights from HuggingFace

`GPT2Model.from_pretrained("gpt2")` downloads the **124 M** parameter checkpoint  
from the HuggingFace Hub on first run (~500 MB) and caches it locally.  
Subsequent runs load from cache instantly.

We use `GPT2Model` (not `GPT2LMHeadModel`) because it gives us direct access  
to the raw transformer weights without an extra wrapper layer.  
The state-dict keys start with `wte`, `wpe`, `h.0.`, `h.1.`, … (no `transformer.` prefix).

In [ ]:
print("Downloading GPT-2 small weights (cached after first run)...")  # user-facing progress message
hf_model = GPT2Model.from_pretrained("gpt2")   # fetch checkpoint; ~500 MB download on first run
hf_model.eval()                                 # put HF model in eval mode (disables HF-side dropout)
print("Done. Model class:", type(hf_model))     # confirm the loaded class type

hf_sd = hf_model.state_dict()                   # extract a flat dict of {name: tensor} for all weights

print("\nFirst 12 weight names and shapes:")     # preview which keys we'll need to map
for k in list(hf_sd.keys())[:12]:               # iterate over first 12 entries in the state dict
    print(f"  {k:<50s} {tuple(hf_sd[k].shape)}")  # left-align the key name, then show shape

## 4. The Conv1D vs nn.Linear Weight Transposition Problem

HuggingFace's GPT-2 uses a custom `Conv1D` class (not `nn.Linear`).  
`Conv1D` stores weights as **`[in_features, out_features]`**,  
whereas `nn.Linear` stores them as **`[out_features, in_features]`**.

This means every weight matrix we copy must be **transposed** with `.T`  
or the matrix multiplication inside `nn.Linear.forward()` will produce wrong results.

The combined QKV matrix makes this concrete:

In [ ]:
c_attn_w = hf_sd["h.0.attn.c_attn.weight"]                   # combined QKV Conv1D weight for layer 0
print("c_attn Conv1D weight shape (in_features, out_features):", c_attn_w.shape)  # expect [768, 2304]

transposed = c_attn_w.T                                        # flip to nn.Linear convention [out, in]
print("After .T — nn.Linear convention (out_features, in_features): ", transposed.shape)  # expect [2304, 768]

q_w, k_w, v_w = transposed.chunk(3, dim=0)                    # split 2304 rows into 3 × 768 rows
print(f"Q weight: {q_w.shape}   K weight: {k_w.shape}   V weight: {v_w.shape}")  # each should be [768, 768]

c_attn_b = hf_sd["h.0.attn.c_attn.bias"]                     # combined QKV bias — 1-D, no transpose needed
print("c_attn bias shape:", c_attn_b.shape)                   # expect [2304]
q_b, k_b, v_b = c_attn_b.chunk(3, dim=0)                      # split bias into 3 × [768]
print(f"Q bias: {q_b.shape}   K bias: {k_b.shape}   V bias: {v_b.shape}")  # each [768]

## 5. Defining `load_gpt2_weights()`

This function iterates over every parameter in our `GPTModel` and fills it  
from the corresponding HuggingFace weight, applying `.T` wherever the Conv1D  
transposition is needed.

We wrap everything in `torch.no_grad()` so that the in-place `.copy_()` calls  
do not create gradient history nodes — we are assigning values, not computing.

In [ ]:
def load_gpt2_weights(gpt_model, hf_model, cfg):
    """Copy weights from a HuggingFace GPT2Model into our custom GPTModel."""

    hf_sd = hf_model.state_dict()     # extract all HuggingFace weights as a flat {name: tensor} dict

    with torch.no_grad():              # disable gradient tracking — we are assigning, not training

        # ── Token and Position Embeddings ─────────────────────────────────────────
        # Embeddings are just lookup tables (2-D matrices); no transposition needed.
        gpt_model.tok_emb.weight.copy_(hf_sd["wte.weight"])   # word token embedding [50257, 768]
        gpt_model.pos_emb.weight.copy_(hf_sd["wpe.weight"])   # word position embedding [1024, 768]

        # ── Final Layer Norm ───────────────────────────────────────────────────────
        # GPT-2 calls them weight/bias; our custom LayerNorm calls them scale/shift.
        gpt_model.final_norm.scale.copy_(hf_sd["ln_f.weight"])  # layer norm gain  (gamma) [768]
        gpt_model.final_norm.shift.copy_(hf_sd["ln_f.bias"])    # layer norm offset (beta)  [768]

        # ── Output Head ───────────────────────────────────────────────────────────
        # GPT-2 ties the language-model head to the token embedding matrix (weight tying).
        # We copy the same wte.weight into out_head so our model benefits from this too.
        gpt_model.out_head.weight.copy_(hf_sd["wte.weight"])   # reuse token emb weights [50257, 768]

        # ── Per-Layer Weights ──────────────────────────────────────────────────────
        for i in range(cfg["n_layers"]):        # loop over all 12 transformer blocks
            blk = gpt_model.trf_blocks[i]      # shorthand reference to block i in our model
            h   = f"h.{i}"                      # HuggingFace key prefix, e.g. "h.0", "h.1", ...

            # Layer Norms — weight → scale, bias → shift; shapes are [768], no transpose needed.
            blk.norm1.scale.copy_(hf_sd[f"{h}.ln_1.weight"])   # pre-attention norm, learnable scale
            blk.norm1.shift.copy_(hf_sd[f"{h}.ln_1.bias"])     # pre-attention norm, learnable shift
            blk.norm2.scale.copy_(hf_sd[f"{h}.ln_2.weight"])   # pre-FFN norm, learnable scale
            blk.norm2.shift.copy_(hf_sd[f"{h}.ln_2.bias"])     # pre-FFN norm, learnable shift

            # ── QKV Projections ────────────────────────────────────────────────────
            # c_attn is a Conv1D with weight [768, 2304].  Transpose → [2304, 768].
            # Then split along dim=0 into Q [768,768], K [768,768], V [768,768].
            c_attn_w = hf_sd[f"{h}.attn.c_attn.weight"].T      # transpose: Conv1D [768,2304]→[2304,768]
            c_attn_b = hf_sd[f"{h}.attn.c_attn.bias"]          # combined bias [2304] — no transpose

            q_w, k_w, v_w = c_attn_w.chunk(3, dim=0)           # split into Q,K,V weights each [768,768]
            q_b, k_b, v_b = c_attn_b.chunk(3, dim=0)           # split into Q,K,V biases  each [768]

            blk.att.W_query.weight.copy_(q_w)                   # load Q projection weight matrix
            blk.att.W_key.weight.copy_(k_w)                     # load K projection weight matrix
            blk.att.W_value.weight.copy_(v_w)                   # load V projection weight matrix

            if cfg["qkv_bias"]:                                 # GPT-2 uses QKV bias; guard for future configs
                blk.att.W_query.bias.copy_(q_b)                 # Q bias vector [768]
                blk.att.W_key.bias.copy_(k_b)                   # K bias vector [768]
                blk.att.W_value.bias.copy_(v_b)                 # V bias vector [768]

            # ── Attention Output Projection (W_o) ──────────────────────────────────
            # c_proj is a square Conv1D [768, 768]; .T gives nn.Linear [768, 768].
            blk.att.W_o.weight.copy_(                           # copy output projection weight
                hf_sd[f"{h}.attn.c_proj.weight"].T              # transpose Conv1D [768,768] → [768,768]
            )
            blk.att.W_o.bias.copy_(hf_sd[f"{h}.attn.c_proj.bias"])   # W_o bias [768]

            # ── Feed-Forward MLP ───────────────────────────────────────────────────
            # layers[0]: expand linear 768 → 3072; layers[1]: GELU; layers[2]: compress 3072 → 768

            blk.ff.layers[0].weight.copy_(                      # expand weight: Conv1D [768,3072]→[3072,768]
                hf_sd[f"{h}.mlp.c_fc.weight"].T                 # .T converts from Conv1D to nn.Linear layout
            )
            blk.ff.layers[0].bias.copy_(hf_sd[f"{h}.mlp.c_fc.bias"])   # expand bias [3072]

            blk.ff.layers[2].weight.copy_(                      # compress weight: Conv1D [3072,768]→[768,3072]
                hf_sd[f"{h}.mlp.c_proj.weight"].T               # .T converts from Conv1D to nn.Linear layout
            )
            blk.ff.layers[2].bias.copy_(hf_sd[f"{h}.mlp.c_proj.bias"]) # compress bias [768]

    print(f"All weights loaded for {cfg['n_layers']} layers.")   # confirmation once the loop finishes

## 6. Instantiating Our Model and Loading the Weights

We first create a `GPTModel` with random weights, then call `load_gpt2_weights()`  
to overwrite every parameter with the pretrained values.

After loading we switch to **eval mode** with `model.eval()`.  
This disables the dropout layers — we never want dropout during inference  
because it randomly zeroes activations and makes output non-deterministic.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"    # use GPU if one is available, else fall back to CPU
print(f"Running on: {device}")                             # confirm which device we are using

gpt = GPTModel(GPT_CONFIG_124M)                            # create our model with randomly initialised weights
gpt = gpt.to(device)                                       # move all tensors (params + buffers) to the device

load_gpt2_weights(gpt, hf_model, GPT_CONFIG_124M)         # overwrite random weights with GPT-2 pretrained values
gpt.eval()                                                  # switch to inference mode: disables dropout

n_params = sum(p.numel() for p in gpt.parameters())        # count total learnable parameters
print(f"Model loaded. Parameters: {n_params / 1e6:.1f} M")  # should print ~124.4 M for GPT-2 small

## 7. Verifying the Weight Transfer

Before generating text, let's do a quick numerical sanity check:  
compare a few of our loaded weights against the HuggingFace reference tensors.

`torch.allclose(a, b, atol=1e-5)` returns `True` if every element of `a` and `b`  
differs by less than `1e-5`.  We use a small tolerance because floating-point  
copy operations on different devices can introduce tiny rounding differences.

In [ ]:
with torch.no_grad():                                                         # disable grad for comparison ops

    # ── Token Embeddings ──────────────────────────────────────────────────────
    our_tok = gpt.tok_emb.weight.cpu()                                        # move our weight to CPU for comparison
    hf_tok  = hf_sd["wte.weight"]                                             # HF reference (already on CPU)
    match_tok = torch.allclose(our_tok, hf_tok, atol=1e-5)                   # element-wise closeness check
    print(f"Token embeddings match:            {match_tok}")                  # should print True

    # ── Layer 0 Q Projection Weight ───────────────────────────────────────────
    our_q0 = gpt.trf_blocks[0].att.W_query.weight.cpu()                      # our W_query for block 0
    hf_q0  = hf_sd["h.0.attn.c_attn.weight"].T[:768]                        # HF Q slice after transpose
    match_q = torch.allclose(our_q0, hf_q0, atol=1e-5)                      # compare numerically
    print(f"Layer 0 Q projection matches:      {match_q}")                   # should print True

    # ── Layer 0 Feed-Forward Expand Weight ────────────────────────────────────
    our_ff0 = gpt.trf_blocks[0].ff.layers[0].weight.cpu()                    # our first MLP linear weight
    hf_ff0  = hf_sd["h.0.mlp.c_fc.weight"].T                                # HF c_fc transposed
    match_ff = torch.allclose(our_ff0, hf_ff0, atol=1e-5)                   # check they match
    print(f"Layer 0 FF expand weight matches:  {match_ff}")                  # should print True

    # ── Final LayerNorm Scale ─────────────────────────────────────────────────
    our_ln = gpt.final_norm.scale.cpu()                                       # our final norm scale vector
    hf_ln  = hf_sd["ln_f.weight"]                                            # HF ln_f.weight
    match_ln = torch.allclose(our_ln, hf_ln, atol=1e-5)                     # compare
    print(f"Final LayerNorm scale matches:     {match_ln}")                  # should print True

## 8. Generating Text with Pretrained Weights

Now the fun part.  We'll use:
- **`tiktoken`** to encode a text prompt into BPE token IDs.
- Our **`generate()`** function from `model.py` to autoregressively extend the sequence.
- **`tiktoken.decode()`** to convert the output IDs back to a readable string.

Generation strategy: **greedy decoding** — at each step we pick the single most  
probable next token.  This is deterministic and produces fluent (if sometimes  
repetitive) text.  Sampling with temperature would give more varied output.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")   # load GPT-2's BPE vocabulary (50,257 tokens); cached after first call

def text_to_token_ids(text, tokenizer):
    """Encode a plain-text string into a (1, seq_len) LongTensor."""
    encoded = tokenizer.encode(text)             # BPE-encode: string → list of integer token IDs
    ids = torch.tensor(encoded).unsqueeze(0)     # convert list to tensor, add batch dim → (1, seq_len)
    return ids                                   # return shape (1, seq_len)

def token_ids_to_text(token_ids, tokenizer):
    """Decode a (1, seq_len) LongTensor back into a plain-text string."""
    flat = token_ids.squeeze(0).tolist()         # remove batch dimension, convert tensor → Python list
    return tokenizer.decode(flat)                # BPE-decode: list of ints → readable string

### Prompt 1: "The meaning of life is"

A classic open-ended philosophical prompt.  With random weights this would produce  
incoherent tokens.  With GPT-2 weights loaded, the model should continue it  
with grammatically correct and contextually plausible English.

In [ ]:
prompt1 = "The meaning of life is"                                              # seed text for generation
input_ids1 = text_to_token_ids(prompt1, tokenizer).to(device)                  # tokenize and move to device

output_ids1 = generate(                                                         # call our greedy decoder
    model          = gpt,                                                       # our GPTModel with GPT-2 weights
    idx            = input_ids1,                                                # token ids of the seed prompt
    max_new_tokens = 40,                                                        # number of new tokens to append
    context_length = GPT_CONFIG_124M["context_length"],                        # crop window size = 1024
)

result1 = token_ids_to_text(output_ids1, tokenizer)                            # decode all token ids to text
print("Prompt  :", prompt1)                                                     # echo the original prompt
print("Output  :", result1)                                                     # show full generated continuation

### Prompt 2: "In a shocking finding, scientists"

A journalistic opening.  GPT-2 was trained heavily on Reddit outbound links  
and news articles, so it should continue in a news-story style.

In [ ]:
prompt2 = "In a shocking finding, scientists"                                   # second seed prompt
input_ids2 = text_to_token_ids(prompt2, tokenizer).to(device)                  # tokenize second prompt

output_ids2 = generate(                                                         # greedy generation
    model          = gpt,                                                       # same pretrained model
    idx            = input_ids2,                                                # token ids of second prompt
    max_new_tokens = 40,                                                        # generate 40 new tokens
    context_length = GPT_CONFIG_124M["context_length"],                        # crop at 1024 to stay in window
)

result2 = token_ids_to_text(output_ids2, tokenizer)                            # decode output ids to string
print("Prompt  :", prompt2)                                                     # echo the prompt
print("Output  :", result2)                                                     # print the full generation

## Summary

We successfully loaded OpenAI's GPT-2 weights into a model built entirely from  
scratch.  The key steps were:

| Step | What we did |
|------|-------------|
| 1 | Defined the exact GPT-2 small config (`emb_dim=768`, `n_heads=12`, `n_layers=12`, `qkv_bias=True`) |
| 2 | Downloaded weights with `GPT2Model.from_pretrained("gpt2")` |
| 3 | Transposed every Conv1D weight with `.T` before copying to `nn.Linear` |
| 4 | Split the fused `c_attn` matrix into separate `W_query`, `W_key`, `W_value` |
| 5 | Verified numerical closeness with `torch.allclose` |
| 6 | Generated text from two prompts using greedy decoding |

**Next steps**: fine-tuning on a domain-specific dataset (Ch 6), or  
benchmarking inference throughput on Jetson Orin / Nano edge hardware.